In [4]:
"""
Simple Unicode CSV Cleaner

Fixes Unicode encoding issues in CSV files with minimal code.
"""

import pandas as pd
import unicodedata
import re
from pathlib import Path
from typing import Optional, List


def clean_text(text):
    """Remove problematic Unicode characters"""
    if not isinstance(text, str):
        return str(text) if text is not None else ""
    
    # Remove surrogates (main cause of encoding errors)
    text = ''.join(c for c in text if not (0xD800 <= ord(c) <= 0xDFFF))
    
    # Remove control characters except tab/newline
    text = ''.join(c for c in text if unicodedata.category(c) != 'Cc' or c in '\t\n\r')
    
    # Normalize and ensure UTF-8 compatibility
    text = unicodedata.normalize('NFKD', text)
    text = text.encode('utf-8', 'ignore').decode('utf-8')
    
    return text


def read_csv_safe(file_path: str, **kwargs):
    """Read CSV with multiple encoding attempts"""
    encodings = ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']
    
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding, **kwargs)
        except UnicodeDecodeError:
            continue
    
    raise ValueError(f"Could not read {file_path} with any encoding")


def save_csv_safe(df, file_path: str, **kwargs):
    """Save CSV with error handling"""
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    
    save_options = [
        {'encoding': 'utf-8', 'errors': 'strict'},
        {'encoding': 'utf-8', 'errors': 'ignore'},
        {'encoding': 'utf-8-sig', 'errors': 'replace'}
    ]
    
    for options in save_options:
        try:
            df.to_csv(file_path, **{**kwargs, **options})
            return True
        except UnicodeEncodeError:
            continue
    
    return False


def clean_csv(file_path: str, columns: Optional[List[str]] = None):
    """
    Clean Unicode issues in CSV file (modifies original file)
    
    Args:
        file_path: CSV file path to clean
        columns: Columns to clean (default: all string columns)
    
    Returns:
        Success status and basic stats
    """
    try:
        # Read CSV
        df = read_csv_safe(file_path)
        
        # Determine columns to clean
        if columns is None:
            columns = df.select_dtypes(include=['object']).columns.tolist()
        
        # Clean specified columns
        for col in columns:
            if col in df.columns:
                df[col] = df[col].apply(clean_text)
        
        # Save back to original file
        success = save_csv_safe(df, file_path, index=False)
        
        return {
            'success': success,
            'rows': len(df),
            'columns_cleaned': len(columns)
        }
        
    except Exception as e:
        return {'success': False, 'error': str(e)}


def clean_multiple_csvs(file_paths: List[str]):
    """Clean multiple CSV files in-place"""
    results = []
    for file_path in file_paths:
        result = clean_csv(file_path)
        result['file'] = file_path
        results.append(result)
    
    return results


# Simple usage examples
if __name__ == "__main__":
    # Clean single file in-place
    dataset = "inspired"
    model = 'llama3-2-1b-instruct'
    alg= 'sft_generated'
    # file = 'conversation_pairs_generated.csv'
    # file = 'preference_pairs_with_rewards_final.csv'
    file = 'train_sft_generated.csv'
    
    result = clean_csv(f"{dataset}/{alg}/{model}/{file}")
    print(f"Cleaned: {result}")
    
    # # Clean specific columns only
    # result = clean_csv("file.csv", columns=['text_col', 'reasoning_col'])
    
    # # Clean multiple files in-place
    # files = ["file1.csv", "file2.csv", "file3.csv"]
    # results = clean_multiple_csvs(files)

Cleaned: {'success': True, 'rows': 544, 'columns_cleaned': 7}
